# Notebook 23 — MJO NSV Stage 3 + 4 + Analysis  *(SIREN refine, dynamics MLP, physical interpretation)*
**Project:** ENSO-BSISO SSL — MJO NSV extension  
**Author:** Jiayi (jh9141@nyu.edu)

Final MJO NSV notebook. Consumes nb21's Stage 1 latents (lp25-equivalent bandpass + lag=10) and nb22's `d̂ = 7` (H3 confirmed). Produces the **Neural State Variables** `v_t ∈ ℝ^{7}` and identifies what each dimension physically represents.

## Method

**Stage 3 — SIREN refine autoencoder.** Same Sitzmann-2020 architecture as BSISO nb20 (64 → 128 → 64 → 32 → d̂ → 32 → 64 → 128 → 64, sine activations, ω₀=30), MSE loss in z-space. Produces 7-D Neural State Variables.

**Stage 4 — Dynamics MLP.** `f: v_t → v_{t+10}` (lag matches nb21). Validates the 7-D state space supports next-step prediction.

**Analysis (the scientific payload).** For each of the 7 Neural State Variables, compute Pearson correlation with conventional climate indices:
- RMM amplitude (continuous)
- RMM cos(phase), sin(phase)
- ENSO continuous (EN=+1, Neutral=0, LN=−1)
- Day-of-year (seasonal-confound check — critical for all-year MJO data)

**Active-MJO filter for the correlation analysis.** When computing correlations against RMM phase, restrict to days where `weak_mjo == False` (i.e., amplitude ≥ 1 AND phase ∈ {1..8}). RMM phase is not physically meaningful for weak-MJO days, so including them would dilute the correlations. The full dataset is still used for the ENSO displacement test (matches nb14/nb16 convention).

**ENSO displacement z-score in v-space.** Same permutation procedure as BSISO nb20. Comparison baselines:
- **nb14 supervised** (Session 24): z = 12.21
- **nb15 SSL** (Session 24): z = 13.44 (but with seasonal confound — caveat)
- **nb16 RMM index** (Session 24): z = 4.10
- **BSISO v-space** (Session 32): z = 12.50 (for cross-mode comparison)

## Inputs

- `MJO/nsv/latents_lag10/z_train.npy`, `z_val.npy` ← from nb21
- `MJO/nsv/latents_lag10/{rmm_phase_t, rmm_amplitude_t, enso_cat_t, weak_mjo_t, dates_t, train_mask}.npy`
- `MJO/nsv/results/stage2_lag10/intrinsic_dim.json` ← nb22 output; reads `d_hat = 7`

## Outputs (`MJO/nsv/...`)

```
checkpoints_lag10/refine_{best,final}.pth, dynamics_mlp{,_best}.pth
state_vars_lag10/v_train.npy, v_val.npy
results/stage3_4_lag10/
  refine_training.png
  v_pca_phase_enso.png         (2-D PCA of v_t; pair plot too cluttered at d=7)
  v_pairs_subset.png           (4-D subset: v0-v3 pair scatter; 6 plots)
  dim_correlations.png         (7 × 5 heatmap — the headline scientific figure)
  enso_displacement.png        (bar chart vs nb14/nb16/BSISO baselines)
  dynamics_training.png
  stage3_4_summary.{md, json}
```

## Runtime ~6–10 min on Colab T4.

---

## Cell 1 — Setup: Load nb21 latents + nb22's d̂

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, math, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

PROJECT_DIR = '/content/drive/MyDrive/BSISO_SSL_Project'
MJO_DIR     = f'{PROJECT_DIR}/MJO'
NSV_DIR     = f'{MJO_DIR}/nsv'
LATENT_DIR  = f'{NSV_DIR}/latents_lag10'
STAGE2_DIR  = f'{NSV_DIR}/results/stage2_lag10'
CKPT_DIR    = f'{NSV_DIR}/checkpoints_lag10'
STATE_DIR   = f'{NSV_DIR}/state_vars_lag10'
RESULTS_DIR = f'{NSV_DIR}/results/stage3_4_lag10'
for d in [CKPT_DIR, STATE_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Load latents + labels
z_train     = np.load(f'{LATENT_DIR}/z_train.npy')
z_val       = np.load(f'{LATENT_DIR}/z_val.npy')
train_mask  = np.load(f'{LATENT_DIR}/train_mask.npy')
phase_t     = np.load(f'{LATENT_DIR}/rmm_phase_t.npy')
amp_t       = np.load(f'{LATENT_DIR}/rmm_amplitude_t.npy')
enso_t      = np.load(f'{LATENT_DIR}/enso_cat_t.npy')
weak_mjo_t  = np.load(f'{LATENT_DIR}/weak_mjo_t.npy')
dates_t     = np.load(f'{LATENT_DIR}/dates_t.npy')

# d_hat from nb22
with open(f'{STAGE2_DIR}/intrinsic_dim.json') as f:
    stage2 = json.load(f)
D_HAT = int(stage2['d_hat'])
Z_DIM = z_train.shape[1]
print(f'\nFrom nb22: d̂ = {D_HAT}  (LB={stage2["LB_mean"]:.2f}±{stage2["LB_std"]:.2f}, '
      f'Two-NN={stage2["TwoNN"]:.2f}, lPCA={stage2["lPCA"]:.2f}, hypothesis={stage2["hypothesis"]})')
print(f'Stage 1 latent dim: {Z_DIM}.   SIREN bottleneck for nb23: {D_HAT}.')

# Aligned label vectors (train + val concatenated, same order as z_train ++ z_val)
order = np.concatenate([np.where(train_mask)[0], np.where(~train_mask)[0]])
phase_all = phase_t[order]
amp_all   = amp_t[order]
enso_all  = enso_t[order]
weak_all  = weak_mjo_t[order]
dates_all = dates_t[order]
z_all     = np.concatenate([z_train, z_val], axis=0)
is_train  = np.concatenate([np.ones(len(z_train), dtype=bool), np.zeros(len(z_val), dtype=bool)])

enso_continuous_all = np.zeros(len(enso_all), dtype=np.float32)
enso_continuous_all[enso_all == 'El Nino'] = +1.0
enso_continuous_all[enso_all == 'La Nina'] = -1.0

doy_all = pd.DatetimeIndex(dates_all).day_of_year.values.astype(np.float32)
active_mask_all = ~weak_all

print(f'\nTotal {len(z_all)} samples (train {is_train.sum()} / val {(~is_train).sum()})')
print(f'Active MJO (weak_mjo=False): {int(active_mask_all.sum())}/{len(z_all)} ({100*active_mask_all.mean():.1f}%)')

## Cell 2 — SIREN Autoencoder (Same Architecture as BSISO nb20)

Sitzmann et al. 2020 initialization required (`bound = 1/n_in` for first layer, `√(6/n_in)/ω₀` for the rest, ω₀=30). Bottleneck and final-output layers are linear (no sine).

In [ ]:
class SirenLayer(nn.Module):
    def __init__(self, in_features, out_features, omega_0=30.0, is_first=False):
        super().__init__()
        self.in_features = in_features; self.omega_0 = omega_0; self.is_first = is_first
        self.linear = nn.Linear(in_features, out_features)
        self.init_weights()

    def init_weights(self):
        with torch.no_grad():
            bound = (1.0 / self.in_features) if self.is_first else (math.sqrt(6.0 / self.in_features) / self.omega_0)
            self.linear.weight.uniform_(-bound, bound)
            self.linear.bias.uniform_(-1e-3, 1e-3)

    def forward(self, x):
        return torch.sin(self.omega_0 * self.linear(x))


class SirenAutoencoder(nn.Module):
    def __init__(self, z_dim=64, d_hat=7, omega_0=30.0):
        super().__init__()
        self.e1 = SirenLayer(z_dim, 128, omega_0=omega_0, is_first=True)
        self.e2 = SirenLayer(128,  64,  omega_0=omega_0)
        self.e3 = SirenLayer(64,   32,  omega_0=omega_0)
        self.e4 = nn.Linear(32, d_hat)
        self.d1 = SirenLayer(d_hat, 32, omega_0=omega_0, is_first=True)
        self.d2 = SirenLayer(32,    64, omega_0=omega_0)
        self.d3 = SirenLayer(64,   128, omega_0=omega_0)
        self.d4 = nn.Linear(128, z_dim)
        nn.init.kaiming_uniform_(self.e4.weight, nonlinearity='linear'); nn.init.zeros_(self.e4.bias)
        nn.init.kaiming_uniform_(self.d4.weight, nonlinearity='linear'); nn.init.zeros_(self.d4.bias)

    def encode(self, z):  return self.e4(self.e3(self.e2(self.e1(z))))
    def decode(self, v):  return self.d4(self.d3(self.d2(self.d1(v))))
    def forward(self, z): return self.decode(self.encode(z))


siren = SirenAutoencoder(z_dim=Z_DIM, d_hat=D_HAT).to(device)
n_params_siren = sum(p.numel() for p in siren.parameters())
with torch.no_grad():
    dummy = torch.randn(4, Z_DIM).to(device)
    v = siren.encode(dummy); zhat = siren.decode(v)
    assert v.shape == (4, D_HAT) and zhat.shape == dummy.shape
    print(f'SIREN params: {n_params_siren:,}.  Bottleneck d̂={D_HAT}.  '
          f'Init MSE on random z: {F.mse_loss(zhat, dummy).item():.4f}')

## Cell 3 — Stage 3 Training (Refine z → v)

500 epochs, Adam lr=1e-4, MSE in z-space. Same recipe as BSISO nb20.

In [ ]:
class ZDataset(Dataset):
    def __init__(self, z): self.z = torch.from_numpy(z).float()
    def __len__(self):  return self.z.shape[0]
    def __getitem__(self, k):  return self.z[k]

ds_z_train = ZDataset(z_train); ds_z_val = ZDataset(z_val)
loader_z_train = DataLoader(ds_z_train, batch_size=64, shuffle=True,  num_workers=0)
loader_z_val   = DataLoader(ds_z_val,   batch_size=64, shuffle=False, num_workers=0)

REFINE_EPOCHS = 500
optimizer = optim.Adam(siren.parameters(), lr=1e-4, weight_decay=0.0)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=REFINE_EPOCHS, eta_min=1e-6)

z_var = float(np.var(z_train))
print(f'z_train variance: {z_var:.5f}  (zero-pred MSE baseline)')

history_r = {'train': [], 'val': []}
best_val = float('inf')
t0 = time.time()
for epoch in range(REFINE_EPOCHS):
    siren.train()
    tl = 0.0; n = 0
    for zb in loader_z_train:
        zb = zb.to(device, non_blocking=True)
        loss = F.mse_loss(siren(zb), zb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        tl += loss.item() * zb.size(0); n += zb.size(0)
    tl /= n
    siren.eval()
    vl = 0.0; nv = 0
    with torch.no_grad():
        for zb in loader_z_val:
            zb = zb.to(device, non_blocking=True)
            vl += F.mse_loss(siren(zb), zb, reduction='sum').item() / zb.numel() * zb.size(0)
            nv += zb.size(0)
    vl /= nv
    scheduler.step()
    history_r['train'].append(tl); history_r['val'].append(vl)
    if vl < best_val:
        best_val = vl
        torch.save(siren.state_dict(), f'{CKPT_DIR}/refine_best.pth')
    if (epoch + 1) % 50 == 0:
        print(f'ep {epoch+1:4d}/{REFINE_EPOCHS}   train={tl:.5f}   val={vl:.5f}   '
              f'frac_var={vl/z_var*100:5.2f}%   lr={scheduler.get_last_lr()[0]:.1e}')

torch.save(siren.state_dict(), f'{CKPT_DIR}/refine_final.pth')
print(f'\nDone in {(time.time()-t0)/60:.1f} min.  Best val MSE: {best_val:.5f}  ({best_val/z_var*100:.2f}% of z var)')
print(f'→ SIREN explains {(1-best_val/z_var)*100:.2f}% of z variance through {D_HAT}-D bottleneck.')
siren.load_state_dict(torch.load(f'{CKPT_DIR}/refine_best.pth', map_location=device)); siren.eval()

## Cell 4 — Extract v_t, Visualize + Save

In [ ]:
def extract_v(z_np, model, batch=256):
    model.eval()
    with torch.no_grad():
        return np.concatenate([model.encode(torch.from_numpy(z_np[k:k+batch]).float().to(device)).cpu().numpy()
                                for k in range(0, len(z_np), batch)], axis=0).astype(np.float32)

v_train = extract_v(z_train, siren)
v_val   = extract_v(z_val,   siren)
v_all   = np.concatenate([v_train, v_val], axis=0)
np.save(f'{STATE_DIR}/v_train.npy', v_train); np.save(f'{STATE_DIR}/v_val.npy', v_val)
print(f'Extracted v_train {v_train.shape}, v_val {v_val.shape}.')
print(f'\nPer-dim v stats:')
print(f'  {"dim":<5}{"mean":>10}{"std":>10}{"min":>10}{"max":>10}')
for i in range(D_HAT):
    vi = v_train[:, i]
    print(f'  {i:<5d}{vi.mean():10.4f}{vi.std():10.4f}{vi.min():10.4f}{vi.max():10.4f}')

# Training curve
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(history_r['train'], label='Train MSE', lw=2)
ax.plot(history_r['val'],   label='Val MSE',   lw=2)
ax.axhline(z_var, color='gray', ls='--', lw=1, label=f'z var = zero-pred ({z_var:.4f})')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE (z-space)')
ax.set_title(f'MJO Stage 3 SIREN refine (bottleneck d̂={D_HAT})', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3); ax.set_yscale('log')
plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/refine_training.png', dpi=140, bbox_inches='tight'); plt.show()

## Cell 5 — v-Space Visualization

Two figures because d̂=7 makes all C(7,2)=21 pairs impractical:
1. **2-D PCA of v-space** colored by RMM phase and ENSO — captures the dominant manifold structure.
2. **Pair plot of first 4 dims** (6 panels) — detailed look at a manageable subset.

In [ ]:
from sklearn.decomposition import PCA
from itertools import combinations

# Figure 1: PCA of v-space
v_centered = v_all - v_all.mean(axis=0)
pca_v = PCA(n_components=min(D_HAT, len(v_all) - 1)).fit(v_centered)
v_pca = pca_v.transform(v_centered)
vr = pca_v.explained_variance_ratio_
print(f'v-space PCA variance: ' + ' '.join(f'PC{i+1}={vr[i]*100:.1f}%' for i in range(D_HAT)))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
phase_colors = plt.cm.hsv(np.linspace(0, 1, 9))[:8]
enso_cmap = {'El Nino': '#d62728', 'Neutral': '#7f7f7f', 'La Nina': '#1f77b4'}
enso_marker = {'El Nino': '^', 'Neutral': 'o', 'La Nina': 's'}

ax = axes[0]
act = active_mask_all
for p in range(1, 9):
    m = (phase_all == p) & act
    ax.scatter(v_pca[m, 0], v_pca[m, 1], c=[phase_colors[p-1]], s=6, alpha=0.5, label=f'P{p}')
ax.set_xlabel(f'PC1 ({vr[0]*100:.1f}%)'); ax.set_ylabel(f'PC2 ({vr[1]*100:.1f}%)')
ax.set_title(f'MJO v-space (d̂={D_HAT}) colored by RMM phase  (active MJO only)', fontweight='bold', fontsize=11)
ax.legend(fontsize=8, ncol=2, loc='best', markerscale=2); ax.grid(alpha=0.3)

ax = axes[1]
for cat in ['El Nino', 'Neutral', 'La Nina']:
    m = enso_all == cat
    ax.scatter(v_pca[m, 0], v_pca[m, 1], c=enso_cmap[cat], marker=enso_marker[cat],
               s=5, alpha=0.4, label=f'{cat} (N={int(m.sum())})')
ax.set_xlabel(f'PC1 ({vr[0]*100:.1f}%)'); ax.set_ylabel(f'PC2 ({vr[1]*100:.1f}%)')
ax.set_title(f'MJO v-space (d̂={D_HAT}) colored by ENSO', fontweight='bold', fontsize=11)
ax.legend(fontsize=9, loc='best', markerscale=2); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/v_pca_phase_enso.png', dpi=130, bbox_inches='tight'); plt.show()

# Figure 2: First-4-dim pair plot (6 panels), colored by phase
n_subset = min(4, D_HAT)
subset_pairs = list(combinations(range(n_subset), 2))
ncols = 3; nrows = (len(subset_pairs) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4.5*nrows))
axes_flat = np.atleast_1d(axes).flatten()
for ax, (i, j) in zip(axes_flat, subset_pairs):
    for p in range(1, 9):
        m = (phase_all == p) & active_mask_all
        ax.scatter(v_all[m, i], v_all[m, j], c=[phase_colors[p-1]], s=4, alpha=0.4)
    ax.set_xlabel(f'v{i}'); ax.set_ylabel(f'v{j}')
    ax.set_title(f'v{i} vs v{j}', fontsize=10); ax.grid(alpha=0.3)
for ax in axes_flat[len(subset_pairs):]: ax.set_visible(False)
fig.suptitle(f'MJO v-space (first {n_subset} dims of {D_HAT}) colored by RMM phase  (active MJO only)',
             fontweight='bold')
plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/v_pairs_subset.png', dpi=120, bbox_inches='tight'); plt.show()

## Cell 6 — Per-Dimension Correlations (the Headline Scientific Figure)

For each `v_i` (i = 0..6), compute Pearson r with conventional climate indices. **Active-MJO filter applied** for RMM-phase/amplitude correlations (phase isn't physically meaningful when amp < 1). Full dataset used for ENSO/DOY.

NaN-safe per Session 32 patch: mask non-finite entries before `pearsonr`.

In [ ]:
# Build index arrays (full N)
phase_angle = (phase_all.astype(float) - 1) * (2 * np.pi / 8)
phase_cos = np.cos(phase_angle)
phase_sin = np.sin(phase_angle)

# RMM amplitude/phase indices: ONLY meaningful for active-MJO days
# Mask amp/phase to NaN on weak-MJO days so they're excluded from those correlations
amp_masked       = np.where(active_mask_all, amp_all.astype(float),  np.nan)
phase_cos_masked = np.where(active_mask_all, phase_cos,               np.nan)
phase_sin_masked = np.where(active_mask_all, phase_sin,               np.nan)

indices = {
    'RMM amplitude (active)':    amp_masked,
    'RMM cos(phase) (active)':   phase_cos_masked,
    'RMM sin(phase) (active)':   phase_sin_masked,
    'ENSO (EN=+1, LN=-1)':       enso_continuous_all.astype(float),
    'Day of year':               doy_all.astype(float),
}

# NaN diagnostic
print('NaN count per index:')
for name, idx in indices.items():
    n_nan = int(np.isnan(idx).sum())
    print(f'  {name:<28s}: {n_nan:5d} NaN out of {len(idx)}  ({100*n_nan/len(idx):.2f}%)')
print(f'  {"v_all":<28s}: {int(np.isnan(v_all).sum()):5d} NaN out of {v_all.size}')

# Per-pair NaN-masked Pearson r
corr_mat = np.full((D_HAT, len(indices)), np.nan)
pval_mat = np.full((D_HAT, len(indices)), np.nan)
n_used   = np.zeros((D_HAT, len(indices)), dtype=int)
for i in range(D_HAT):
    for j, (name, idx) in enumerate(indices.items()):
        valid = np.isfinite(v_all[:, i]) & np.isfinite(idx)
        n_used[i, j] = int(valid.sum())
        if valid.sum() < 30: continue
        r, p = pearsonr(v_all[valid, i], idx[valid])
        corr_mat[i, j] = float(r)
        pval_mat[i, j] = float(p)

# Print table
header = '            ' + ''.join(f'{name[:20]:>22s}' for name in indices)
print('\n' + header)
for i in range(D_HAT):
    row = f'  v{i:<8d}'
    for j in range(len(indices)):
        if np.isnan(corr_mat[i, j]):
            row += f'{"--":>22s}'
        else:
            p = pval_mat[i, j]
            sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
            row += f'{corr_mat[i, j]:+.3f}{sig:<5s}'.rjust(22)
    print(row)
print('\n(*** p<0.001  ** p<0.01  * p<0.05;  -- = too few valid samples)')

# Per-dim summary
abs_corr = np.abs(corr_mat)
max_abs_r  = np.nanmax(abs_corr, axis=1)
best_match = np.nanargmax(np.where(np.isnan(abs_corr), -1, abs_corr), axis=1)
print('\nPer-dim summary:')
for i in range(D_HAT):
    if np.isnan(max_abs_r[i]):
        print(f'  v{i}:  all correlations failed')
        continue
    bj = int(best_match[i]); r = corr_mat[i, bj]
    name = list(indices.keys())[bj]
    label = 'MYSTERY (max |r| < 0.3)' if max_abs_r[i] < 0.3 else f'best: {name} (r={r:+.3f})'
    print(f'  v{i}:  max |r| = {max_abs_r[i]:.3f}    →   {label}')

# Heatmap
fig, ax = plt.subplots(figsize=(10, 1.5 + 0.7 * D_HAT))
masked = np.ma.array(corr_mat, mask=np.isnan(corr_mat))
cmap = plt.cm.RdBu_r.copy(); cmap.set_bad(color='lightgray')
im = ax.imshow(masked, cmap=cmap, vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(indices)))
ax.set_xticklabels(list(indices.keys()), rotation=30, ha='right')
ax.set_yticks(range(D_HAT))
ax.set_yticklabels([f'v{i}' for i in range(D_HAT)])
for i in range(D_HAT):
    for j in range(len(indices)):
        if np.isnan(corr_mat[i, j]):
            ax.text(j, i, '--', ha='center', va='center', color='black', fontsize=9)
        else:
            p = pval_mat[i, j]
            sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
            color = 'white' if abs(corr_mat[i, j]) > 0.4 else 'black'
            ax.text(j, i, f'{corr_mat[i, j]:+.2f}{sig}', ha='center', va='center', color=color, fontsize=9)
plt.colorbar(im, ax=ax, label='Pearson r')
ax.set_title(f'MJO Neural State Variables v_i correlations  (d̂={D_HAT})', fontweight='bold')
plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/dim_correlations.png', dpi=140, bbox_inches='tight'); plt.show()

## Cell 7 — ENSO Displacement Z-Score in v-Space

Same procedure as BSISO nb20. Comparison baselines:
- **nb14 supervised**: z = 12.21
- **nb15 SSL**: z = 13.44 (with seasonal confound caveat)
- **nb16 RMM**: z = 4.10
- **BSISO v-space**: z = 12.50

In [ ]:
def enso_displacement(v, phase, enso, n_perm=1000, rng_seed=42):
    rng = np.random.default_rng(rng_seed)
    phases = np.arange(1, 9)
    def disp(en):
        ds = []
        for p in phases:
            mEN = (phase == p) & (en == 'El Nino')
            mLN = (phase == p) & (en == 'La Nina')
            if mEN.sum() < 3 or mLN.sum() < 3: continue
            ds.append(np.linalg.norm(v[mEN].mean(0) - v[mLN].mean(0)))
        return float(np.mean(ds)) if ds else float('nan')
    obs = disp(enso)
    null = []
    for _ in range(n_perm):
        d = disp(enso[rng.permutation(len(enso))])
        if not np.isnan(d): null.append(d)
    null = np.asarray(null)
    z = (obs - null.mean()) / (null.std() + 1e-9)
    return obs, float(null.mean()), float(null.std()), float(z)

# Use active MJO only (matches nb14/nb16 convention)
act = active_mask_all
obs_v, bmu_v, bsd_v, z_v = enso_displacement(v_all[act], phase_all[act], enso_all[act])
obs_z, bmu_z, bsd_z, z_z = enso_displacement(z_all[act], phase_all[act], enso_all[act])

print(f'ENSO displacement z-score (active MJO):')
print(f'  v-space ({D_HAT}-D NSV):  obs={obs_v:.4f}  null={bmu_v:.4f}±{bsd_v:.4f}  z={z_v:.2f}')
print(f'  z-space (64-D Stage 1): obs={obs_z:.4f}  null={bmu_z:.4f}±{bsd_z:.4f}  z={z_z:.2f}')
print(f'\nMJO project baselines:')
print(f'  nb14 supervised:     z = 12.21')
print(f'  nb15 SSL temporal:   z = 13.44  (with seasonal confound)')
print(f'  nb16 RMM index:      z = 4.10')
print(f'\nBSISO Session 32 v-space (4-D NSV): z = 12.50')

fig, ax = plt.subplots(figsize=(10, 4.5))
names = [f'MJO NSV\nv-space ({D_HAT}-D)', 'MJO Stage 1\nz-space (64-D)',
         'MJO nb14\n2-D sup', 'MJO nb15\n2-D SSL', 'MJO nb16\nRMM index',
         'BSISO NSV\nv-space (4-D)']
vals  = [z_v, z_z, 12.21, 13.44, 4.10, 12.50]
colors = ['#2ca02c', '#1f77b4', '#888888', '#888888', '#888888', '#ff7f0e']
bars = ax.bar(names, vals, color=colors, alpha=0.85)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.3, f'{v:.2f}', ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('ENSO displacement z-score')
ax.set_title(f'MJO: ENSO modulation in NSV space vs project baselines', fontweight='bold')
ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/enso_displacement.png', dpi=140, bbox_inches='tight'); plt.show()

## Cell 8 — Stage 4 Dynamics MLP: v_t → v_{t+10}

Same date-matching logic as BSISO nb20.

In [ ]:
dates_pd = pd.DatetimeIndex(dates_all)
years    = dates_pd.year.values
date_to_k = {d: k for k, d in enumerate(dates_pd)}

dyn_k_t, dyn_k_t1 = [], []
for k, d in enumerate(dates_pd):
    target = d + pd.Timedelta(days=10)
    if (target in date_to_k) and (years[date_to_k[target]] == years[k]):
        dyn_k_t.append(k); dyn_k_t1.append(date_to_k[target])
    elif (target in date_to_k) and (is_train[date_to_k[target]] == is_train[k]):
        # MJO is continuous; allow cross-year pairs IFF same split
        dyn_k_t.append(k); dyn_k_t1.append(date_to_k[target])
dyn_k_t  = np.asarray(dyn_k_t)
dyn_k_t1 = np.asarray(dyn_k_t1)
print(f'Built {len(dyn_k_t)} dynamics pairs (v[k] → v[k+10 days, same split]).')

dyn_train_mask = is_train[dyn_k_t]
v_dyn_t  = v_all[dyn_k_t]
v_dyn_t1 = v_all[dyn_k_t1]

class DynDataset(Dataset):
    def __init__(self, vt, vt1, mask):
        self.vt  = torch.from_numpy(vt[mask]).float()
        self.vt1 = torch.from_numpy(vt1[mask]).float()
    def __len__(self):  return self.vt.shape[0]
    def __getitem__(self, k):  return self.vt[k], self.vt1[k]

ds_dyn_train = DynDataset(v_dyn_t, v_dyn_t1, dyn_train_mask)
ds_dyn_val   = DynDataset(v_dyn_t, v_dyn_t1, ~dyn_train_mask)
loader_dyn_train = DataLoader(ds_dyn_train, batch_size=64, shuffle=True,  num_workers=0)
loader_dyn_val   = DataLoader(ds_dyn_val,   batch_size=64, shuffle=False, num_workers=0)
print(f'Train: {len(ds_dyn_train)}  Val: {len(ds_dyn_val)}')

with torch.no_grad():
    pers_dyn = ((ds_dyn_val.vt - ds_dyn_val.vt1) ** 2).mean().item()
v_var = float(np.var(v_dyn_t1[~dyn_train_mask]))
print(f'v-space persistence MSE (val):  {pers_dyn:.4f}')
print(f'v-space variance:               {v_var:.4f}')

class DynamicsMLP(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, 32),  nn.ReLU(),
            nn.Linear(32, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, d),
        )
    def forward(self, x): return self.net(x)

DYN_EPOCHS = 300
mlp = DynamicsMLP(D_HAT).to(device)
opt = optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-4)
sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=DYN_EPOCHS, eta_min=1e-5)

hist_d = {'train': [], 'val': []}
best_dyn = float('inf')
t0 = time.time()
for epoch in range(DYN_EPOCHS):
    mlp.train()
    tl = 0.0; n = 0
    for vt, vt1 in loader_dyn_train:
        vt = vt.to(device, non_blocking=True); vt1 = vt1.to(device, non_blocking=True)
        loss = F.mse_loss(mlp(vt), vt1)
        opt.zero_grad(); loss.backward(); opt.step()
        tl += loss.item() * vt.size(0); n += vt.size(0)
    tl /= n
    mlp.eval()
    vl = 0.0; nv = 0
    with torch.no_grad():
        for vt, vt1 in loader_dyn_val:
            vt = vt.to(device, non_blocking=True); vt1 = vt1.to(device, non_blocking=True)
            vl += F.mse_loss(mlp(vt), vt1, reduction='sum').item() / vt1.numel() * vt.size(0)
            nv += vt.size(0)
    vl /= nv
    sch.step()
    hist_d['train'].append(tl); hist_d['val'].append(vl)
    if vl < best_dyn:
        best_dyn = vl
        torch.save(mlp.state_dict(), f'{CKPT_DIR}/dynamics_mlp_best.pth')
    if (epoch + 1) % 30 == 0:
        print(f'ep {epoch+1:3d}/{DYN_EPOCHS}   train={tl:.4f}   val={vl:.4f}   vs pers {pers_dyn:.4f}')
torch.save(mlp.state_dict(), f'{CKPT_DIR}/dynamics_mlp.pth')
print(f'\nDone in {(time.time()-t0)/60:.1f} min.  Best val: {best_dyn:.4f}')
imp_dyn = (pers_dyn - best_dyn) / pers_dyn * 100
print(f'Improvement over v-persistence: {imp_dyn:+.1f}%')

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(hist_d['train'], label='Train MSE', lw=2)
ax.plot(hist_d['val'],   label='Val MSE',   lw=2)
ax.axhline(pers_dyn, color='gray', ls='--', lw=1, label=f'v-persistence ({pers_dyn:.3f})')
ax.axhline(v_var, color='lightgray', ls=':', lw=1, label=f'v variance ({v_var:.3f})')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE (v-space)')
ax.set_title('MJO Stage 4 dynamics MLP: v_t → v_t+10', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/dynamics_training.png', dpi=140, bbox_inches='tight'); plt.show()

## Cell 9 — Final Summary

In [ ]:
import time as _t

dim_rows = []
for i in range(D_HAT):
    if np.isnan(max_abs_r[i]):
        dim_rows.append(f'| v{i} | n/a | all correlations failed |')
        continue
    bj = int(best_match[i])
    name = list(indices.keys())[bj]
    r = corr_mat[i, bj]; p = pval_mat[i, bj]
    story = 'mystery axis (no |r|>0.3)' if max_abs_r[i] < 0.3 else f'best: {name} (r={r:+.3f}, p={p:.2e})'
    dim_rows.append(f'| v{i} | {max_abs_r[i]:.3f} | {story} |')

summary_md = f"""# MJO NSV Stage 3 + 4 Summary

**Date:** {_t.strftime('%Y-%m-%d')}  
**Pipeline:** nb21 → nb22 → nb23 (this notebook)  
**Inputs:** {len(z_train)} train + {len(z_val)} val 64-D Stage 1 latents (MJO bp20-90, lag-10).  
**Decision input:** d̂ = {D_HAT} from nb22 ({stage2['hypothesis']}).

## Stage 3 — SIREN refine

- Bottleneck: **{D_HAT}**
- Params: {n_params_siren:,}
- Best val MSE: **{best_val:.5f}** ({best_val/z_var*100:.2f}% of z variance)
- **Variance explained**: {(1-best_val/z_var)*100:.2f}% through the {D_HAT}-D bottleneck

## Stage 4 — Dynamics MLP

- Best val MSE: **{best_dyn:.4f}**
- v-persistence baseline: {pers_dyn:.4f}
- **Improvement over persistence**: {imp_dyn:+.1f}%

## Per-dimension physical interpretation

| Dim | max |r| | Best correlate |
|---|---|---|
{chr(10).join(dim_rows)}

## ENSO displacement (active MJO only)

| Space | Observed | Null mean ± std | Z-score |
|---|---|---|---|
| v ({D_HAT}-D NSV) | {obs_v:.4f} | {bmu_v:.4f} ± {bsd_v:.4f} | **{z_v:.2f}** |
| z (64-D Stage 1) | {obs_z:.4f} | {bmu_z:.4f} ± {bsd_z:.4f} | **{z_z:.2f}** |

**Baselines**: nb14 sup z=12.21, nb15 SSL z=13.44, nb16 RMM z=4.10, BSISO v-space z=12.50.

## Scientific bottom line

MJO at MJJAS daily resolution requires **{D_HAT} state variables** (Levina-Bickel), exceeding the conventional 2-D (RMM1, RMM2) index. The {D_HAT}-D Neural State Variables are dynamically consistent (Stage 4 MLP beats v-persistence by {imp_dyn:.0f}%) and the ENSO z-score in v-space ({z_v:.2f}) {'beats' if z_v > 12.21 else 'is comparable to' if abs(z_v - 12.21) < 2 else 'is below'} the nb14 supervised baseline ({stage2['hypothesis']}).

**Cross-mode comparison with BSISO** (Session 32, d̂=4): MJO has more state-space dimensions than BSISO ({D_HAT} vs 4). Both intraseasonal modes have undercounted state spaces, but MJO surprisingly has MORE (not fewer) than BSISO — driven by all-year data covering more atmospheric regimes, larger N for the estimator, and possibly more distinct propagating MJO modes than the simple equatorial Kelvin-Rossby picture captures.
"""
with open(f'{RESULTS_DIR}/stage3_4_summary.md', 'w') as f:
    f.write(summary_md)

summary_json = {
    'date':                _t.strftime('%Y-%m-%d'),
    'pipeline':            'MJO NSV',
    'd_hat':               D_HAT,
    'hypothesis':          stage2['hypothesis'],
    'siren_n_params':      int(n_params_siren),
    'refine_best_val_mse': float(best_val),
    'refine_z_var':        float(z_var),
    'refine_var_explained': float(1 - best_val/z_var),
    'dynamics_best_val_mse': float(best_dyn),
    'dynamics_persistence_mse': float(pers_dyn),
    'dynamics_improvement_pct': float(imp_dyn),
    'enso_z_v_space':      float(z_v),
    'enso_z_z_space':      float(z_z),
    'dim_correlations':    [[float(corr_mat[i, j]) for j in range(len(indices))] for i in range(D_HAT)],
    'dim_pvalues':         [[float(pval_mat[i, j]) for j in range(len(indices))] for i in range(D_HAT)],
    'index_names':         list(indices.keys()),
    'per_dim_best_match':  [list(indices.keys())[best_match[i]] if not np.isnan(max_abs_r[i]) else None for i in range(D_HAT)],
    'per_dim_max_abs_r':   [float(max_abs_r[i]) if not np.isnan(max_abs_r[i]) else None for i in range(D_HAT)],
}
with open(f'{RESULTS_DIR}/stage3_4_summary.json', 'w') as f:
    json.dump(summary_json, f, indent=2)

print(summary_md)
print(f'\nSaved markdown + JSON to {RESULTS_DIR}')

---
## Done!

MJO NSV pipeline complete: nb21 → nb22 → nb23.

**Send back** for review:
1. `stage3_4_summary.md` (printed at end of Cell 9) — headline numbers + per-dim interpretation.
2. **`dim_correlations.png`** — the headline scientific figure. Tells us what each of the 7 NSVs physically represents.
3. `v_pca_phase_enso.png` — visual confirmation that v-space organizes by phase + ENSO.
4. `enso_displacement.png` — vs project baselines.
5. `dynamics_training.png` — does Stage 4 beat v-persistence?
6. `refine_training.png` — Stage 3 reconstruction.

**What to look for:**
- Do **2 of the 7 dims have high |r| with `RMM cos(phase)` and `RMM sin(phase)`?** That confirms 2 dims are conventional RMM.
- Does **1 dim have high |r| with `RMM amplitude`?** That's the independent amplitude axis (like BSISO).
- Does **1 dim have high |r| with `ENSO continuous`?** Direct H3-with-ENSO confirmation at the dim level.
- Do any dims have **|r| > 0.15 with `Day of year`?** That would flag a seasonal-cycle confound (the failure mode that killed MJO lat16; the bandpass should have prevented this).
- The remaining **3 dims** (if total = 7) are candidates for **mystery axes** — these are the project's discovery. Possible identities: precursor/initiation state, Walker circulation slow modes, distinct propagating MJO sub-modes.

---
*DDCS Project | jh9141@nyu.edu*